In [1]:
import pandas as pd
import numpy as np
import pyreadr
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from IPython.display import HTML

In [2]:
# Read from CSV; load RData
esg_tq   = pd.read_csv("public/ESGscore_TobinsQ.csv")
esg_data = list(pyreadr.read_r("public/ESG_data.RData").values())[0]

In [3]:
# Merge and engineer features
df = (
    esg_data
    .assign(date=lambda x: x["date"].astype(str))
    .merge(esg_tq, left_on=["instrument", "date"], right_on=["tic", "datadate"], how="left")
    .assign(
        at_pos    = lambda x: x["at"].where(x["at"] > 0),
        ceq_pos   = lambda x: x["ceq"].where(x["ceq"] > 0),
        q         = lambda x: (x["prcc_f"] * x["csho"] + x["dltt"] + x["dlc"]) / x["at_pos"],
        ESGscore  = lambda x: x["esg_metric"],
        Firm_Size = lambda x: np.log(x["at"]),
        Leverage  = lambda x: (x["dltt"] + x["dlc"]) / x["ceq_pos"],
        Year      = lambda x: x["fyear"],
        Industry  = lambda x: x["sich"],
    )
    [["source", "q", "ESGscore", "Firm_Size", "Leverage", "Year", "Industry"]]
    .dropna()
)
df_B = df[df["source"] == "Provider_B"].drop(columns="source").reset_index(drop=True)
df_A = df[df["source"] == "Provider_A"].drop(columns="source").reset_index(drop=True)

In [4]:
# Summary statistics
df_B.describe()

,q,ESGscore,Firm_Size,Leverage,Year,Industry
count,7895.000000,7895.000000,7895.000000,7895.000000,7895.000000,7895.000000
mean,1.982023,53.304807,9.844767,1.413020,2013.706903,4739.437619
std,1.864408,20.150539,1.471820,5.378326,5.949387,1851.187210
min,0.043002,2.457145,5.582631,0.000000,2001.000000,100.000000
25%,0.917131,37.332651,8.819907,0.351147,2009.000000,3510.000000
50%,1.469155,55.215821,9.791610,0.720379,2014.000000,4911.000000
75%,2.388747,69.895601,10.692390,1.321548,2019.000000,6311.000000
max,23.269175,95.162371,15.170158,264.722222,2023.000000,9997.000000


In [5]:
# Descriptive statistics table
cols = ["q", "ESGscore", "Firm_Size", "Leverage"]
(
    df_B[cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .T
    .rename(columns={"count": "N", "mean": "Mean", "median": "Median",
                     "std": "SD", "min": "Min", "max": "Max"})
    .assign(N=lambda x: x["N"].astype(int))
    .round(3)
)

,N,Mean,Median,SD,Min,Max
q,7895,1.982,1.469,1.864,0.043,23.269
ESGscore,7895,53.305,55.216,20.151,2.457,95.162
Firm_Size,7895,9.845,9.792,1.472,5.583,15.170
Leverage,7895,1.413,0.720,5.378,0.000,264.722


In [6]:
# Correlation table
df_B[["q", "ESGscore", "Firm_Size", "Leverage"]].corr().round(3)

,q,ESGscore,Firm_Size,Leverage
q,1.000,-0.043,-0.439,0.007
ESGscore,-0.043,1.000,0.442,0.059
Firm_Size,-0.439,0.442,1.000,0.056
Leverage,0.007,0.059,0.056,1.000


In [7]:
# Fit all regression models
m1      = smf.ols("q ~ ESGscore", data=df_B).fit()
m2      = smf.ols("q ~ ESGscore + Firm_Size + Leverage", data=df_B).fit()
m3      = smf.ols("q ~ ESGscore + Firm_Size + Leverage + C(Industry)", data=df_B).fit()
m4      = smf.ols("q ~ ESGscore + Firm_Size + Leverage + C(Industry) + C(Year)", data=df_B).fit()
m_alt   = smf.ols("q ~ ESGscore + Firm_Size + Leverage + C(Industry) + C(Year)", data=df_A).fit()
m_large = smf.ols("q ~ ESGscore + Firm_Size + Leverage + C(Industry) + C(Year)",
                  data=df_B[df_B["Firm_Size"] > df_B["Firm_Size"].median()]).fit()
m_1019  = smf.ols("q ~ ESGscore + Firm_Size + Leverage + C(Industry) + C(Year)",
                  data=df_B[df_B["Year"].between(2010, 2019)]).fit()

In [8]:
# Main results table
sg = Stargazer([m1, m2, m3, m4])
sg.title("MAIN RESULTS")
sg.dependent_variable_name("q")
sg.custom_columns(["Model 1", "Model 2", "Model 3", "Model 4"], [1, 1, 1, 1])
sg.covariate_order(["ESGscore", "Firm_Size", "Leverage"])
sg.rename_covariates({"Firm_Size": "Firm Size"})
sg.add_line("Industry Indicators", ["No", "No", "Yes", "Yes"])
sg.add_line("Year Indicators",     ["No", "No", "No",  "Yes"])
sg.show_degrees_of_freedom(False)
HTML(sg.render_html())

In [9]:
# Robustness analysis table
sg2 = Stargazer([m4, m_alt, m_large, m_1019])
sg2.title("ROBUSTNESS ANALYSIS")
sg2.dependent_variable_name("q")
sg2.custom_columns(["Main Results", "Alt. ESG Score", "Large Firms", "2010-2019"], [1, 1, 1, 1])
sg2.covariate_order(["ESGscore"])
sg2.add_line("Controls",            ["Yes", "Yes", "Yes", "Yes"])
sg2.add_line("Industry Indicators", ["Yes", "Yes", "Yes", "Yes"])
sg2.add_line("Year Indicators",     ["Yes", "Yes", "Yes", "Yes"])
sg2.show_degrees_of_freedom(False)
HTML(sg2.render_html())